<div class="alert alert-info">
    
➡️ Before you start, make sure that you are familiar with the **[study guide](https://liu-nlp.ai/text-mining/logistics/)**, in particular the rules around **cheating and plagiarism** (found in the course memo).

➡️ If you use code from external sources (e.g. StackOverflow, ChatGPT, ...) as part of your solutions, don't forget to add a reference to these source(s) (for example as a comment above your code).

➡️ Make sure you fill in all cells that say **`YOUR CODE HERE`** or **YOUR ANSWER HERE**.  You normally shouldn't need to modify any of the other cells.

</div>

# L1: Information Retrieval

In this lab you will apply basic techniques from information retrieval to implement the core of a minimalistic search engine. The data for this lab consists of a collection of app descriptions scraped from the [Google Play Store](https://play.google.com/store/apps?hl=en). From this collection, your search engine should retrieve those apps whose descriptions best match a given query under the vector space model.

In [20]:
# Define some helper functions that are used in this notebook
from IPython.display import display, HTML

def success():
    display(HTML('<div class="alert alert-success"><strong>Checks have passed!</strong></div>'))

## Dataset

The app descriptions come in the form of a compressed [JSON](https://en.wikipedia.org/wiki/JSON) file. Start by loading this file into a [Pandas](https://pandas.pydata.org) [DataFrame](https://pandas.pydata.org/pandas-docs/stable/getting_started/dsintro.html#dataframe).

In [21]:
import bz2
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 500)

with bz2.open('app-descriptions.json.bz2', mode='rt', encoding='utf-8') as source:
    df = pd.read_json(source, encoding='utf-8')

In Pandas, a DataFrame is a table with indexed rows and labelled columns of potentially different types. You can access data in a DataFrame in various ways, including by row and column. To give an example, the code in the next cell shows rows 200–204:

In [22]:
df.loc[200:205]

,name,description
200,Brick Breaker Star: Space King,"Introducing the best Brick Breaker game that everyone can enjoy.\nEnjoy various missions and addictively simple play control.\n\n[Features]\n- Hundreds of stages and various missions\n- No limit to play such as Heart, play as much as you can!\n- 5 kinds of various items and items reinforcement system\n- No network required\n- game file is as low as 20M, light-weight download!\n- supports tablet screen\n- supports Google Play Leaderboards, Achievement, Multiplay\n- supports 14 languages\n\nHo..."
201,Brick Classic - Brick Game,"Classic Brick Game!\n\nBrick Classic is a popular and addictive puzzle game!\n\nHow to play?\n- Simply drag the bricks to move them.\n- Create full lines on the grid vertically or horizontally to break bricks.\n\nTips:\n- Classic brick game without time limits.\n- Place the bricks in a reasonable position.\n- The more brick break, the more scores you have.\n- Bricks can't be rotated.\n\nWho's the best brick breaker? Challenge it now!!!"
202,Bricks Breaker - Glow Balls,"Bricks Breaker - Glow Balls is a addictive and challenging brick game.\nJust play it to relax your brain. Be focus on breaking bricks and you will find it more funny and exciting.\n\nHow to play\n- Hold the screen with your finger and move to aim.\n- Find best positions and angles to hit all bricks.\n- When the durability of brick reaches 0, destroyed.\n- Never let bricks reach the bottom or game is over.\n\nFeatures\n- Colorful glow skins.\n- Free to play.\n- Easy game controls with one fin..."
203,Bricks Breaker Quest,"How to play\n- The ball flies to wherever you touched.\n- Clear the stages by removing bricks on the board.\n- Break the bricks and never let them hit the bottom.\n- Find best positions and angles to hit every brick.\n\nFeature\n- Free to play\n- Tons of stages\n- Various types of balls\n- Easy to play, Simplest game system, Designed for one handheld gameplay.\n- Off-line (without internet connection) gameplay supported \n- Multi-play supported\n- Tablet device supported\n- Achievement & lea..."
204,Brothers in Arms® 3,"Fight brave soldiers from around the globe on the frenzied multiplayer battlegrounds of World War 2 or become Sergeant Wright and experience a dramatic, life-changing single-player journey, in the aftermath of the D-Day invasion.\n\nCLIMB THE ARMY RANKS IN MULTIPLAYER \n> 4 maps to master and enjoy. \n> 2 gameplay modes to begin with: Free For All and Team Deathmatch.\n> Unlock game-changing perks by playing with each weapon class!\n> A soldier’s only as deadly as his weapon. Be sure to upgr..."
205,Brown Dust - Tactical RPG,"The Empire has fallen, and the Age of Great Mercenaries Now Begins!\nCreate Your Ultimate Team And Strike Down Your Enemies!\n\nCAPTIVATING AND STUNNING ARTWORK\n- Experience the high-quality anime illustrations you have never seen before.\n- Meet Brown Dust's charming Mercenaries now.\n\nASSEMBLE LEGENDARY MERCENARIES\n- Over 300 Mercenaries and a Variety of Skills.\n- Discover the Unique Mercenaries, 6 Devils and Dominus Octo.\n- All Mercenaries can reach max level and the highest rank.\n\..."


As you can see, there are two labelled columns: `name` (the name of the app) and `description` (a textual description). The next cell shows how to access only the description field from row 200:

In [23]:
df.loc[200, 'description']

'Introducing the best Brick Breaker game that everyone can enjoy.\nEnjoy various missions and addictively simple play control.\n\n[Features]\n- Hundreds of stages and various missions\n- No limit to play such as Heart, play as much as you can!\n- 5 kinds of various items and items reinforcement system\n- No network required\n- game file is as low as 20M, light-weight download!\n- supports tablet screen\n- supports Google Play Leaderboards, Achievement, Multiplay\n- supports 14 languages\n\nHomepage:\nhttps://play.google.com/store/apps/dev?id=4931745640662708567\n\nFacebook: \nhttps://www.facebook.com/spcomesgames/'

## Problem 1: What's in a vector?

We start by vectorising the data — more specifically, we map each app description to a tf–idf vector. This is very simple with a library like [scikit-learn](https://scikit-learn.org/stable/), which provides a [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) class for exactly this purpose.  If we instantiate this class, and call `fit_transform()` on all of our app descriptions, scikit-learn will preprocess and tokenize each app description, compute tf–idf values for each of them, and return a vectorised representation:

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()



X = vectorizer.fit_transform(df['description'])
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 267110 stored elements and shape (1614, 27877)>

Let’s pick the app "Pancake Tower", which has a rather short description text, to see how it has been vectorised:

In [25]:
# We can use 'toarray' to convert the sparse matrix object into a "normal" array
vec = X[1032].toarray()[0]

# The app description & its corresponding vector
df.loc[1032, 'description'], vec

("Let's see how many pancakes you can pile up!!",
 array([0., 0., 0., ..., 0., 0., 0.], shape=(27877,)))

That's not very informative yet.  We know that the vector contains tf–idf values, and that each dimension of the vector corresponds to a token in the vectorizer’s vocabulary; let's extract these for this specific example.

Your **first task** is to find out how to access the `vectorizer`’s vocabulary, for example by [checking the documentation of `TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html), and print all the tokens that are represented in the vector with a tf–idf value greater than zero (i.e., only the tokens that are actually part of this app’s description) _in descending order of the tf–idf values_.  In other words, the token with the highest tf–idf value should be at the top of your output, and the token with the lowest tf–idf value at the bottom.   Before you implement this, think about what you would expect the output look like, for example which words you would expect to have the highest/lowest tf–idf values in this example.

Your final output should look something like this:

```
<token 1>: <tf-idf value 1>
<token 2>: <tf-idf value 2>
...
```

In [26]:
tokens = vectorizer.get_feature_names_out()

token_tfidf = [(t, v) for t, v in zip(tokens, vec)]
token_tfidf.sort(key=lambda x: x[1], reverse=True)

for t, v in token_tfidf:
    print(f"{t}: {v}")

pancakes: 0.6539332651185913
pile: 0.5304701435508047
let: 0.2615287714771797
see: 0.2557630827415271
many: 0.23491959669849022
how: 0.21153246225085887
up: 0.17216837691451817
can: 0.13047602895910532
you: 0.10276923239718011
00: 0.0
000: 0.0
0000: 0.0
007: 0.0
01: 0.0
02: 0.0
03: 0.0
05: 0.0
06: 0.0
0771: 0.0
08: 0.0
080: 0.0
09: 0.0
0b6up9: 0.0
0ghz: 0.0
0kr: 0.0
10: 0.0
100: 0.0
1000: 0.0
10000000: 0.0
1000den: 0.0
100125201589035673885: 0.0
100539321026946285107: 0.0
100544026660028: 0.0
100k: 0.0
100s: 0.0
100th: 0.0
101: 0.0
1010: 0.0
1024: 0.0
1028: 0.0
1028호: 0.0
105: 0.0
107456884368395361653: 0.0
108: 0.0
1080p: 0.0
109: 0.0
10cm: 0.0
10k: 0.0
10m: 0.0
10s: 0.0
10th: 0.0
10x10: 0.0
11: 0.0
110: 0.0
111: 0.0
112: 0.0
113: 0.0
113681410008759: 0.0
115: 0.0
115000952828: 0.0
115005748829: 0.0
115005748929: 0.0
115521255825330665764: 0.0
115863800042796476976: 0.0
1177: 0.0
117783509607924: 0.0
118: 0.0
119: 0.0
11v11: 0.0
12: 0.0
120: 0.0
1200: 0.0
123: 0.0
123kidsfun: 0.0
123k

## Problem 2: Finding the nearest vectors

To build a small search engine, we need to be able to turn _queries_ (for example the string "pile up pancakes") into _query vectors_, and then find out which of our app description vectors are closest to the query vector.

For the first part (turning queries into query vectors), we can simply re-use the `vectorizer` that we used for the app descriptions. For the second part, an easy way to find the closest vectors is to use scikit-learn’s [NearestNeighbors](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html) class. This class needs to be _fit_ on a set of vectors (the "training set"; in our case the app descriptions) and can then be used with any vector to find its _nearest neighbors_ in the vector space.

**First,** instantiate and fit a class that returns the _ten (10)_ nearest neighbors:

In [27]:
"""Instantiate and fit a class that returns the 10 nearest neighboring vectors."""

# YOUR CODE HERE
from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(n_neighbors=10, metric = "cosine")
nn.fit(X)

,n_neighbors,10
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


**Second,** implement a function that uses the vectorizer and the fitted class to find the nearest neighbours for a given query string:

In [28]:
def search(query):
    """Find the nearest neighbors in `df` for a query string.

    Arguments:
      query (str): A query string.

    Returns:
      The 10 apps (with name and description) most similar (in terms of
      cosine similarity) to the given query as a Pandas DataFrame.
    """

    X = vectorizer.transform([query])
    d, index = nn.kneighbors(X)

    return df.take(index[0])[["name", "description"]]


### 🤞 Test your code

Test your implementation by running the following cell, which will sanity-check your return value and show the 10 best search results for the query _"pile up pancakes"_:

In [29]:
"""Check that searching for "pile up pancakes" returns a DataFrame with ten results,
   and that the top result is "Pancake Tower"."""

result = search('pile up pancakes')
display(result)
assert isinstance(result, pd.DataFrame), "search() function should return a Pandas DataFrame"
assert len(result) == 10, "search() function should return 10 search results"
assert result.iloc[0]["name"] == "Pancake Tower", "Top search result should be 'Pancake Tower'"
success()

,name,description
1032,Pancake Tower,Let's see how many pancakes you can pile up!!
326,Cooking School: Games for Girls,"Children like to help their parents. They especially like to help with cooking . When there is a cooking in the kitchen, it is no way to play. But cooking is a complicated process and often it ends up with a huge mess in the kitchen. But what if you are so eager to cook pancakes, cake or cupcakes? How to cook all that without doing a cleaning after? We have a solution! Home Cooking School with our curious Hippo has opened especially for parents and children! We do not only cook food here. We..."
656,"Hell’s Cooking — crazy chef burger, kitchen fever","⭐ ⭐ ⭐ ⭐ ⭐ New world of crazy cooking is here. Feel what it means to be a master chef who prepares fantastic fast food in a prominent king kitchen! If you haven't ever tried yourself as a hamburger chef cook, it's possibly the best time for making diner. Download and launch Hell's Cooking — crazy chef burger, kitchen fever HD game and get prepared to jump into a fever and adventurous perfect world of burgers.\n\nNew girls game Hell's Cooking gives you lots of opportunities for your crazy cafe..."
1235,Solitaire,"Solitaire Free by Solitaire Card Games is the #1 klondike solitaire games on android. The solitaire Free is popular and classic card games you know and love.\n\nWe carefully designed a fresh solitaire free modern look, woven into the wonderful solitaire classic feel that everyone loves. \n\nExperience the crisp, clear, and easy to read cards, simple and quick animations, and subtle sounds, in either landscape or portrait views. \n\nYou can move cards with a single tap or drag them to their d..."
1164,Rummy - Free,"Play the famous Rummy card game on your Android Smartphone or Tablet !! \n\nPlay rummy with 2, 3, or 4 players against simulated opponents playing with high-level artificial intelligence. \nThere are a number of rules that can be modified, making this game very faithful to the original. \n\n*** MANY VARIATIONS INCLUDED *** \n\nMany rummy variations are included in the application: \n\n- From 2 to 4 players. \n- Choose the AI level of opponents. \n- Number of cards dealt to each player (from ..."
1181,Sago Mini Trucks and Diggers,"Drive a dump truck with Rosie the hamster! Pile dirt high and dig deep in the ground with diggers, cranes and bulldozers. Build a home for a new friend! Choose a barn, a castle or even a cupcake-house. Don’t forget to add the finishing touches for the proud owner.\n\nOn this construction site, kids love being the boss. With six mighty machines and piles of dirt, you can build all day! Part of the award-winning suite of Sago Mini apps, this app puts kids in charge.\n\nSago Mini apps have no i..."
436,Dr. Panda's Ice Cream Truck,"Chocolate? Vanilla? Strawberry? All three!? You decide! In Dr. Panda’s Ice Cream Truck you can mix up all sorts of different flavors with cookies, chocolate, nuts and more to make the perfect ice cream—hundreds of combinations in all.\n\nScoop it!\nThese animals love ice cream, and will eat as much (or little) as you want to serve them. You can make scoops big or small and pile them as high as you want—using any of the ice cream you’ve created!\n\nToppings galore!\nUse chocolate syrup, cooki..."
1442,Turbo Dismount™,"The legendary crash simulator is now on Google Play!\n\nPerform death-defying motor stunts, crash into walls, create traffic pile-ups of epic scale - and share the fun!\n\nTurbo Dismount™ is a kinetic tragedy about Mr. Dismount and the cars who love him. It is the official sequel to the wildly popular and immensely successful personal impact simulator - Stair Dismount™. \n\nFEATURES:\n* Flinch-inducing crash physics\n* Crunchy sound effects\n* Delicious slow-mo replay system\n* Multiple vehi..."
1446,UNO!™,"Play the world’s number one card game like never before. UNO!™ has all-new rules, tournaments, adventures and so much more! At home or on the move, jump into games instantly. Whet

Before continuing with the next problem, play around a bit with this simple search functionality by trying out different search queries, and see if the results look like what you would expect:

In [30]:
# Example — try out your own queries!
search("dodge trains")

,name,description
1428,Train Conductor World,"Master and manage the chaos of international railway traffic as the ultimate railroad tycoon. Build the rail network of your dreams; lay rails and solve the railroad puzzle with branching and forking roads at every turn. Become the richest manager and pick your path, do you optimise to the micro level, planning routes and managing the timetable, or sit idle letting your business keep earning while you sleep! \n\nGet in the driver's seat and take passengers to their destinations, dropping the..."
1301,Subway Surfers,"DASH as fast as you can! \nDODGE the oncoming trains! \n\nHelp Jake, Tricky & Fresh escape from the grumpy Inspector and his dog. \n\n★ Grind trains with your cool crew! \n★ Colorful and vivid HD graphics! \n★ Hoverboard Surfing! \n★ Paint powered jetpack! \n★ Lightning fast swipe acrobatics! \n★ Challenge and help your friends! \n\nJoin the most daring chase! \n\nA Universal App with HD optimized graphics.\n\nBy Kiloo and Sybo."
1300,Subway Princess Runner,"Subway princess runner, Bus run, forest rush with addictive endless running game!\nRush as fast as you can, dodge the oncoming trains and buses. Careful the rolling wood in the forest! Intuitive controls to run left or right, jump in the sky to obtain more coins, excited slide to safety!\n\nHelp your loved beautiful princess to escape the police! Use skateboard after double tapping, experience the unique board in the subway. Challenge the highest score of the rank with the world players or s..."
998,No Humanity - The Hardest Game,"2M+ Downloads All Over The World!\n\n* IGN Nominated Best Aussie/NZ game *\n* Top 5 indie games at PAX 2015 Australia – Mashable *\n* Global Game Jam ""Best Game"" Sydney 2015 *\n* Global Game Jam ""Best Audio"" Sydney 2015 *\n\nIt's the end of the world and you are the lone survivor in a tiny spaceship. Get ready to dodge everything that is trying to kill you! Your reaction time and precision is key! No Humanity is the hardest bullet hell dodge game. Compare your score with friends and watch as..."
228,Bus Rush 2,"Bus Rush 2 is one of the most complete multiplayer runners for Android. \nRun along Rio de Janeiro and other scenarios. Drag to jump or slide and to move left or right, avoid hitting obstacles like trucks, buses and subway trains among others!\nPlay races with other users around the world in the multiplayer mode. Run around and gather all the coins you can in different scenarios from Rio city like downtown, subway, sewer, forest, different beaches, and an amazing jungle!\n\nIn Bus Rush 2, yo..."
1465,Virus War - Space Shooting Game,"Warning! Virus invasion! Destroy them with your fingertip! \nis a free casual shooting game. Using only your fingertip, destroy all sorts of viruses. Remember to dodge, don’t let those filthy things hit your ship!\n*Simple and engaging gameplay. Play Virus War anywhere and anytime; get the most fun out of your breaks!\n*Equip your ship with different weapons and blast through swarms of enemies!\n*Surpass your friends in the ranking; set new records!"
360,Dancing Road: Color Ball Run!,Try out the most exciting Running - Sliding - Matching Music Game!\n\nThe rolling ball starts simply and ramps up shortly. \n\n★ Hold and drag your rolling ball to match other balls of the same color!\n★ Dodge different color balls!\n★ Try to collect all the coins and Gift Boxes on the dancing road!\n\nEnjoy the catchy music and challenges designed for each dancing road. \n\nLet's roll the ball and feel the beat in this Color Matching Game!
184,Bob - jigsaw puzzles free games for kids & parents,"Free jigsaw puzzles for kids, hundreds of puzzles for toddlers to assemble. Try now kids puzzle games for toddlers.\nJigsaw puzzles are great game for your toddler to play in waiting room or anywhere while you have to wait.\n\nFeatures:\n- kid puzzle game for free\n- unlimited number of pictures, many colorful pictures to choose from by your children\n- Perfect kid game when you 

## Problem 3: Custom preprocessing & tokenization

In Problem 1, you should have seen that `TfidfVectorizer` already performs some preprocessing by default and also does its own tokenization of the input data. This is great for getting started, but often we want to have more control over these steps. We can customize some aspects of the preprocessing through arguments when instantiating `TfidfVectorizer`, but for this exercise, we want to do _all_ of our preprocessing & tokenizing outside of scikit-learn.

Concretely, we want to use [spaCy](https://spacy.io), a library that we will make use of in later labs as well.  Here is a brief example of how to load and use a spaCy model:

In [31]:
import spacy
# Load the small English model, disabling some components that we don't need right now
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner', 'textcat'])

# Take an example sentence and print every token from it separately
doc = nlp("Apple is looking at buying U.K. startup for $1 billion")
for token in doc:
    print(token.text)

Apple
is
looking
at
buying
U.K.
startup
for
$
1
billion


**Your task** is to write a preprocessing function that uses spaCy to perform the following steps:
- tokenization
- lemmatization
- stop word removal
- removing tokens containing non-alphabetical characters

We recommend that you go through the [Linguistic annotations](https://spacy.io/usage/spacy-101#annotations) section of the spaCy&nbsp;101, which demonstrates how you can get the relevant kind of information via the spaCy library.

Implement your preprocessor by completing the following function:

In [32]:
import spacy


doc = nlp("Apple is looking at buying U.K. startup for $1 billion")
for token in doc:
    print(token.text)
print(type(doc))

Apple
is
looking
at
buying
U.K.
startup
for
$
1
billion
<class 'spacy.tokens.doc.Doc'>


In [33]:
def preprocess(text):
    """Preprocess the given text by tokenising it, removing any stop words, 
    replacing each remaining token with its lemma (base form), and discarding 
    all lemmas that contain non-alphabetical characters.

    Arguments:
      text (str): The text to preprocess.

    Returns:
      The list of remaining lemmas after preprocessing (represented as strings).
    """
    #tokenized:
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(text)
    processed = []
    # lemmatization jumped, jumping, jump becomes jump

    # stop word removal removes the, is, at... common noninformative words

    # removes special characters
    
    for token in doc:
      if token.is_alpha and not token.is_stop:
         processed.append(token.lemma_)
      

    return processed




### 🤞 Test your code

Test your implementation by running the following cell:

In [34]:
"""Check that the preprocessing returns the correct output for a number of test cases."""

assert (
    preprocess('Apple is looking at buying U.K. startup for $1 billion') ==
    ['Apple', 'look', 'buy', 'startup', 'billion']
)
assert (
    preprocess('"Love Story" is a country pop song written and sung by Taylor Swift.') ==
    ['Love', 'Story', 'country', 'pop', 'song', 'write', 'sing', 'Taylor', 'Swift']
)
success()

## Problem 4: The effect of preprocessing

To make use of the new `preprocess` function from Problem 3, we need to make sure that we incorporate it into `TfidfVectorizer` and disable all preprocessing & tokenization that `TfidfVectorizer` performs by default. Afterwards, we also need to re-fit the vectorizer and the nearest-neighbors class. To make this a bit easier to handle, let’s take everything we have done so far and put it in a single class `AppSearcher`.

### Task 4.1

**Your first task** is to complete the stub of the `AppSearcher` class given below. Keep in mind:
- The `fit()` function should fit both the vectorizer (from Problem 1) and the nearest-neighbors class (from Problem 2).  Make sure to modify the call to `TfidfVectorizer` to _disable all preprocessing & tokenization_ that it would do by default, and replace it with a call to the `preprocess()` function _defined in `AppSearcher`_.
- For the `preprocess()` function, you can start by copying your solution from Problem 3.
- For the `search()` function, you can copy your solution from Problem 2.
- Make sure to adapt your code to store the everything (data, vectorizer, nearest-neighbors class) within the `AppSearcher` class, so that your solution is independent of the code you wrote above!

In [35]:

# YOUR CODE HERE
from sklearn.neighbors import NearestNeighbors

class AppSearcher:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")


    def fit(self, df):
        """Instantiate and fit all the classes required for the search engine (cf. Problems 1 and 2)."""
        self.df = df

        processed = df["description"].apply(self.preprocess)

        X = vectorizer.fit_transform(processed)

        self.nn = NearestNeighbors(n_neighbors=10, metric = "cosine")
        self.nn.fit(X)

        

    def preprocess(self, text):
        """Preprocess the given text (cf. Problem 3)."""
            #tokenized:
        doc = self.nlp(text)
        processed = []
        # lemmatization jumped, jumping, jump becomes jump

        # stop word removal removes the, is, at... common noninformative words

        # removes special characters
        
        for token in doc:
            if token.is_alpha and not token.is_stop:
                processed.append(token.lemma_)
        

        return " ".join(processed)

    def search(self, query):
        """Find the nearest neighbors in `df` for a query string (cf. Problem 2)."""
        """Find the nearest neighbors in `df` for a query string.

        Arguments:
        query (str): A query string.

        Returns:
        The 10 apps (with name and description) most similar (in terms of
        cosine similarity) to the given query as a Pandas DataFrame.
        """

        X = vectorizer.transform([query])
        d, index = self.nn.kneighbors(X)

        return self.df.take(index[0])[["name", "description"]]

#### 🤞 Test your code

The following cell demonstrates how your class should be used. Note that it can take a bit longer to train it on the data as before, since we’re now calling spaCy for the preprocessing.

In [36]:
apps = AppSearcher()
apps.fit(df)
apps.search("pile up pancakes")

,name,description
1032,Pancake Tower,Let's see how many pancakes you can pile up!!
1181,Sago Mini Trucks and Diggers,"Drive a dump truck with Rosie the hamster! Pile dirt high and dig deep in the ground with diggers, cranes and bulldozers. Build a home for a new friend! Choose a barn, a castle or even a cupcake-house. Don’t forget to add the finishing touches for the proud owner.\n\nOn this construction site, kids love being the boss. With six mighty machines and piles of dirt, you can build all day! Part of the award-winning suite of Sago Mini apps, this app puts kids in charge.\n\nSago Mini apps have no i..."
565,Fruit Cube Blast,"Tap, blast, crush, kaboom! Fruit Cube Blast is an exciting game about exploding colorful blocks and making crazy matches in a wild world of amazing characters!\n\nEmbark upon a fantastic adventure as you solve tons of juicy puzzles. All you have to do is tap to match the colored fruit blocks on the screen. Make bigger matches and you'll create sweet power-ups, perfect for making some fresh explosions across the screen! Clear each puzzle by blasting the pieces away, then use your matching ski..."
1235,Solitaire,"Solitaire Free by Solitaire Card Games is the #1 klondike solitaire games on android. The solitaire Free is popular and classic card games you know and love.\n\nWe carefully designed a fresh solitaire free modern look, woven into the wonderful solitaire classic feel that everyone loves. \n\nExperience the crisp, clear, and easy to read cards, simple and quick animations, and subtle sounds, in either landscape or portrait views. \n\nYou can move cards with a single tap or drag them to their d..."
1263,Spider Solitaire,"Spider Solitaire was built to offer card players a fun way to play their favorite classic in both portrait and landscape mode.\n\nWith large cards and a unique stacking system our Spider card game doesn't have problems fitting your screen like many others do. \n\n* How to play *\n\nTo win a game of spider solitaire, all cards must be removed from the table. Assembling the cards in the tableau allows for cards to be placed in their respective stacks in order. At the beginning of each game, 54..."
1442,Turbo Dismount™,"The legendary crash simulator is now on Google Play!\n\nPerform death-defying motor stunts, crash into walls, create traffic pile-ups of epic scale - and share the fun!\n\nTurbo Dismount™ is a kinetic tragedy about Mr. Dismount and the cars who love him. It is the official sequel to the wildly popular and immensely successful personal impact simulator - Stair Dismount™. \n\nFEATURES:\n* Flinch-inducing crash physics\n* Crunchy sound effects\n* Delicious slow-mo replay system\n* Multiple vehi..."
121,Balloon Paradise - Halloween Games Puzzle Match 3,"Ready to start matching and popping some balloons? Balloon Paradise is a colorful and exciting match 3 puzzle game that takes place in a world high above the clouds! Swap and match shiny balloons and unleash fantabulous power-ups as you work to save the sky city of Floatopia!\n\nBelle's amazing adventure has taken her to a world she never knew existed. Far above the land, hidden by the clouds, is a city made entirely out of balloons. A great gust of wind has stirred up trouble in this beauti..."
1142,Renegade Racing,"The hit online game with OVER 180 MILLION PLAYS has made its way to mobile, bigger and crazier than ever before -- and now with multiplayer!\n\nRenegade Racing is an adrenaline-filled, wacky multiplayer racing game. Perform epic stunts to gain turbo and race your way to victory!\n\nGravity-bending worlds, crazy unlockable cars, awesome power-ups and a whole lot of action awaits.\n\nCompete in 1v5 multiplayer races and climb your way up the arena tiers. Each new tier you unlock will give you ..."
1407,Tomb of the Mask,"Tomb of the Mask is an arcade game with an infinite procedurally generated vertical labyrinth. Seeking for adventure you get into a tomb where you find a strange mask. You put it on and sudden

### Task 4.2

**Your second task** is to experiment with the effect of using (or not using) different preprocessing steps.  We always need to _tokenize_ the text, but other preprocessing steps are optional and require a conscious decision whether to use them or not, such as:
- lemmatization
- lowercasing all characters
- removing stop words
- removing tokens containing non-alphabetical characters

**Modify the definition of the `preprocess()` function** of `AppSearcher` to include/exclude individual preprocessing steps, run some searches, and observe if and how the results change.  Which search queries you try out is up to you — you could compare searching for "pile up pancakes" with "pancake piling", for example; or you could try entirely different search queries aimed at different kinds of apps.  (You can modify the class directly by changing the cell above under Task 4.1, or copy the definitions to the cells below, whichever you prefer; there is no separate code to show for this task, but you will use your observations here for the individual reflection.)

In [ ]:
# same preprocess step
print(preprocess("jumping, is fe8 rgdgobfsots, sdfdhi, eaf8"))

# some junk words do not dissappear
apps.search("jumping, is fe8 robfsots, sdfhi, eaf8")


['jump', 'rgdgobfsot', 'sdfdhi']


,name,description
702,Incredible Jack: Jumping & Running,"An incredible journey in the style of the best console games. Blow up everything around and prevail against 7 BIG bosses. Improve your hero by making him stronger in this awesome platformer.\n\n""It is the best game I have ever played. If you are thinking of getting a game that is the one.""\n\n""It's an enjoyable jumping game with cute graphics and addictive gameplay.""\n\n""Would rate it ★★★★★★ six stars if I could! Very nice and challenging platform game.""\n\n""Very nice challenging adventure g..."
680,Horse Riding Tales - Ride With Friends,"THE MULTIPLAYER HORSE RIDING GAME OF YOUR DREAMS.\n\nBegin your journey from lowly stable hand to dressage super star! Join our horse loving online world where you can customise your character, tame beautiful horses, bring them to your stable and train them for challenging show jumping championships.\n\nWELCOME TO MEADOWCROFT\n\nA small, centuries old town surrounded by rolling fields and vast, open plains. Sleepy. Rustic. Idyllic. Full of animals big and small. A paradise of peace and tranq..."
1403,Toca Life: Stable,"Welcome to a world of horses and adventure! Groom horses, care for them and show them! Engage in some medieval jousting or role-play the Wild West. Ride motorcycles and camp out under the stars! You can play your way in Toca Life: Stable!\n\nEXPLORE AN EXCITING WORLD\nTHE STABLE: Step into the stable and can care for your horses. Wash them, brush them, then saddle up for a trot. When you're finished grooming your horses (and cleaning up their poop!), take a well-deserved break in the club ro..."
649,Helix Crush,"Helix Crush combines one-finger control gameplay with a carefully selected soundtrack to create an experience you have ever been through before. Featuring different stage of increasing difficulty to unlock new skins and popular songs. The key is to not let your color ball drops on obstacles and traps, but also play to the music. Listen to the beat and the melody, and afterward you will find yourself singing and loving this music game.\n\nCome on! Test your reflexes and rhythm skills! See the..."
1382,Thomas Was Alone,"- Pocket Gamer Awards 2015 Nominee: Android Game of the Year\n- BAFTA 2013 Winner: Performance (Danny Wallace as Narrator)\n- BAFTA 2013 Nominee: Story\n- BAFTA 2013 Nominee: Original Score\n\nMeet Thomas, a curious rectangle, and help him unravel the mysteries of his existence in a wonderfully strange world. Experience 100 levels of award-winning platform gameplay and take part in an emotional story narrated by Danny Wallace.\n\n“Across the game’s gaps and hazards, there’s a real emotional ..."
304,Color Hop 3D,"Color Hop 3D combines one-finger control gameplay with a carefully selected EDM songs to create an experience you have ever seen before. The key is to not let your ball hop on the wrong color tiles. Listen to the beat and the melody, and afterward, you will find yourself loving this music game.\n\nCome on! Test your reflexes and hop skills! See the surprises and wonders every level holds for you. Color Hop 3D is a ball game that lets ball hop on the tiles until the end of the color road. Thi..."
498,Equilab - Equestrian Tracker,"The equestrian app which helps riders to reach their goals together with their horses. An app for horse riding that automatically tracks the horse’s movements and gives it back as valuable information to the rider. A new way to improve horse management. Add your stable, horses and connect with your co-riders. Come closer to your equestrian team and connect with your horse friends, coach, farrier, veterinarian and trainer. Plan and organize your equestrian life in a horse customized calendar ..."
1599,دردشاتي - تعارف شات و زواج,"تطبيق دردشاتي هو أضخم وأكبر تطبيق تعارف عربي في العالم العربي !\n\nدردشاتي الذي يشمل أكثر من مليون مستخدم يبحثون عن صداقة تعارف وزواج بالاضافة الى زواج مسيار، وهو تطبيق التعارف العربي الأكبر والاضخم بعدد مسجلين حقيقيين من انحاء العالم 

In [38]:
apps.search("jumping")

,name,description
702,Incredible Jack: Jumping & Running,"An incredible journey in the style of the best console games. Blow up everything around and prevail against 7 BIG bosses. Improve your hero by making him stronger in this awesome platformer.\n\n""It is the best game I have ever played. If you are thinking of getting a game that is the one.""\n\n""It's an enjoyable jumping game with cute graphics and addictive gameplay.""\n\n""Would rate it ★★★★★★ six stars if I could! Very nice and challenging platform game.""\n\n""Very nice challenging adventure g..."
680,Horse Riding Tales - Ride With Friends,"THE MULTIPLAYER HORSE RIDING GAME OF YOUR DREAMS.\n\nBegin your journey from lowly stable hand to dressage super star! Join our horse loving online world where you can customise your character, tame beautiful horses, bring them to your stable and train them for challenging show jumping championships.\n\nWELCOME TO MEADOWCROFT\n\nA small, centuries old town surrounded by rolling fields and vast, open plains. Sleepy. Rustic. Idyllic. Full of animals big and small. A paradise of peace and tranq..."
1403,Toca Life: Stable,"Welcome to a world of horses and adventure! Groom horses, care for them and show them! Engage in some medieval jousting or role-play the Wild West. Ride motorcycles and camp out under the stars! You can play your way in Toca Life: Stable!\n\nEXPLORE AN EXCITING WORLD\nTHE STABLE: Step into the stable and can care for your horses. Wash them, brush them, then saddle up for a trot. When you're finished grooming your horses (and cleaning up their poop!), take a well-deserved break in the club ro..."
649,Helix Crush,"Helix Crush combines one-finger control gameplay with a carefully selected soundtrack to create an experience you have ever been through before. Featuring different stage of increasing difficulty to unlock new skins and popular songs. The key is to not let your color ball drops on obstacles and traps, but also play to the music. Listen to the beat and the melody, and afterward you will find yourself singing and loving this music game.\n\nCome on! Test your reflexes and rhythm skills! See the..."
1382,Thomas Was Alone,"- Pocket Gamer Awards 2015 Nominee: Android Game of the Year\n- BAFTA 2013 Winner: Performance (Danny Wallace as Narrator)\n- BAFTA 2013 Nominee: Story\n- BAFTA 2013 Nominee: Original Score\n\nMeet Thomas, a curious rectangle, and help him unravel the mysteries of his existence in a wonderfully strange world. Experience 100 levels of award-winning platform gameplay and take part in an emotional story narrated by Danny Wallace.\n\n“Across the game’s gaps and hazards, there’s a real emotional ..."
304,Color Hop 3D,"Color Hop 3D combines one-finger control gameplay with a carefully selected EDM songs to create an experience you have ever seen before. The key is to not let your ball hop on the wrong color tiles. Listen to the beat and the melody, and afterward, you will find yourself loving this music game.\n\nCome on! Test your reflexes and hop skills! See the surprises and wonders every level holds for you. Color Hop 3D is a ball game that lets ball hop on the tiles until the end of the color road. Thi..."
498,Equilab - Equestrian Tracker,"The equestrian app which helps riders to reach their goals together with their horses. An app for horse riding that automatically tracks the horse’s movements and gives it back as valuable information to the rider. A new way to improve horse management. Add your stable, horses and connect with your co-riders. Come closer to your equestrian team and connect with your horse friends, coach, farrier, veterinarian and trainer. Plan and organize your equestrian life in a horse customized calendar ..."
1599,دردشاتي - تعارف شات و زواج,"تطبيق دردشاتي هو أضخم وأكبر تطبيق تعارف عربي في العالم العربي !\n\nدردشاتي الذي يشمل أكثر من مليون مستخدم يبحثون عن صداقة تعارف وزواج بالاضافة الى زواج مسيار، وهو تطبيق التعارف العربي الأكبر والاضخم بعدد مسجلين حقيقيين من انحاء العالم 

In [44]:

# YOUR CODE HERE
from sklearn.neighbors import NearestNeighbors

class AppSearcher:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")


    def fit(self, df):
        """Instantiate and fit all the classes required for the search engine (cf. Problems 1 and 2)."""
        self.df = df

        processed = df["description"].apply(self.preprocess)

        X = vectorizer.fit_transform(processed)

        self.nn = NearestNeighbors(n_neighbors=10, metric = "cosine")
        self.nn.fit(X)

        

    def preprocess(self, text):
        """Preprocess the given text (cf. Problem 3)."""
            #tokenized:
        doc = self.nlp(text)
        processed = []
        # lemmatization jumped, jumping, jump becomes jump

        # stop word removal removes the, is, at... common noninformative words

        # removes special characters
        
        for token in doc:
            if token.is_alpha and not token.is_stop:
                processed.append(token.text)
        

        return " ".join(processed)

    def search(self, query):
        """Find the nearest neighbors in `df` for a query string (cf. Problem 2)."""
        """Find the nearest neighbors in `df` for a query string.

        Arguments:
        query (str): A query string.

        Returns:
        The 10 apps (with name and description) most similar (in terms of
        cosine similarity) to the given query as a Pandas DataFrame.
        """

        X = vectorizer.transform([query])
        d, index = self.nn.kneighbors(X)

        return self.df.take(index[0])[["name", "description"]]

In [45]:
#removed lemmatization, still same result

apps = AppSearcher()
apps.fit(df)
apps.search("jumping")

,name,description
1208,Shuriken Jump,Travel with the shuriken as high as possible! This jumping upwards game will test your reaction speed and hopping skills. You will be challenged to leap from beam to beam while slicing fruits.
702,Incredible Jack: Jumping & Running,"An incredible journey in the style of the best console games. Blow up everything around and prevail against 7 BIG bosses. Improve your hero by making him stronger in this awesome platformer.\n\n""It is the best game I have ever played. If you are thinking of getting a game that is the one.""\n\n""It's an enjoyable jumping game with cute graphics and addictive gameplay.""\n\n""Would rate it ★★★★★★ six stars if I could! Very nice and challenging platform game.""\n\n""Very nice challenging adventure g..."
680,Horse Riding Tales - Ride With Friends,"THE MULTIPLAYER HORSE RIDING GAME OF YOUR DREAMS.\n\nBegin your journey from lowly stable hand to dressage super star! Join our horse loving online world where you can customise your character, tame beautiful horses, bring them to your stable and train them for challenging show jumping championships.\n\nWELCOME TO MEADOWCROFT\n\nA small, centuries old town surrounded by rolling fields and vast, open plains. Sleepy. Rustic. Idyllic. Full of animals big and small. A paradise of peace and tranq..."
545,Flip Trickster - Parkour Simulator,"Feel the adrenaline as you are flipping in the air after jumping down some crazy heights.\nDo insane jumps and flips down obstacles and land in the goal area in this parkour and freerunning inspired simulation game. \n\nUsing simple game mechanics you can jump, flip, and crash with only one finger.\n\nFeatures:\nRealistic physics\nParkour inspired gameplay\n10 Stages with over 40 levels\nCharacter customization\n10+ hats to choose from\nMore levels coming soon!\n\nNew levels and content are ..."
1348,Temple Run 2,"With over a zillion downloads, Temple Run redefined mobile gaming. Now get more of the exhilarating running, jumping, turning and sliding you love in Temple Run 2!\n\nNavigate perilous cliffs, zip lines, mines and forests as you try to escape with the cursed idol. How far can you run?!\n\nFEATURES\n\n★ Beautiful new graphics\n★ Gorgeous new organic environments\n★ New obstacles\n★ More powerups\n★ More achievements\n★ Special powers for each character\n★ Bigger monkey!!!\n\nBecome a fan of T..."
677,Hop Ball 3D,"Become a ball games master with Hop Ball 3D - one of the latest music games of 2019. Good news? It's free! 🎧\n\nWhat’s new?\nNew popular musi songs added. Now you can play various types of music online, from the best rhythm Piano to guitar songs, Rock and EDM.\n\nHow to play:\nYour only challenge is to keep the ball jumping on the music tiles while listening to the beat drops, but it's surely not an easy music game. Don't miss the tiles! Follow the rhythm, be the one in control, create majes..."
304,Color Hop 3D,"Color Hop 3D combines one-finger control gameplay with a carefully selected EDM songs to create an experience you have ever seen before. The key is to not let your ball hop on the wrong color tiles. Listen to the beat and the melody, and afterward, you will find yourself loving this music game.\n\nCome on! Test your reflexes and hop skills! See the surprises and wonders every level holds for you. Color Hop 3D is a ball game that lets ball hop on the tiles until the end of the color road. Thi..."
1403,Toca Life: Stable,"Welcome to a world of horses and adventure! Groom horses, care for them and show them! Engage in some medieval jousting or role-play the Wild West. Ride motorcycles and camp out under the stars! You can play your way in Toca Life: Stable!\n\nEXPLORE AN EXCITING WORLD\nTHE STABLE: Step into the stable and can care for your horses. Wash them, brush them, then saddle up for a trot. When you're finished grooming your horses (and cleaning up their poop!), take a well-deserved break in the club ro..."
731,Jungle Adventures 3,"Jungle Adventures 3 - If you 

In [46]:
# jumped produced very different results, it resulted in arabic, russian text which also appear when searching a blank text
print(apps.preprocess("jumped"))
apps.search("jumped")


jumped


,name,description
373,Death Worm™,"""A-a-a-a-ah! He-e-elp! That huge terrible worm just swallowed Grandma! A-a-a-a-ah! Now it's right behind me! Noooo!..."" CRACK!\n...\n""I rode my bike along West twenty-seventh street and then it jumped out right from the ground! Dunno what it is but it's BIG! I hardly could hide myself behind a recycle bin, but it smashed my bike!""\n...\n""Operator, unusual activity reported ten miles east from the town by route twenty. Witnesses say they saw something like a big black snake that appeared from..."
1599,دردشاتي - تعارف شات و زواج,"تطبيق دردشاتي هو أضخم وأكبر تطبيق تعارف عربي في العالم العربي !\n\nدردشاتي الذي يشمل أكثر من مليون مستخدم يبحثون عن صداقة تعارف وزواج بالاضافة الى زواج مسيار، وهو تطبيق التعارف العربي الأكبر والاضخم بعدد مسجلين حقيقيين من انحاء العالم العربي .\n\nنقدم لكم تطبيق التعارف الاقوى والاشهر في الشرق الاوسط تطبيق ""دردشاتي""، والمعد بصورة خاصة للمستخدمين العرب، حيث يشمل التطبيق مئات الاف المشتركين ، والمئات الجدد يوميا من شتى انحاء العالم العربي .\n\nالتطبيق تم برمجته وتصميمه بأفضل النواحي التقنية حيث..."
1598,جدول شرح در متن : جدولستان,اگر علاقه زیادی به جدول دارید و دوست دارید در هر مکانی جدول حل کنید، به جدولستان خوش آمدید.\nسرزمینی جذاب که پر است از جدول های سرگرم کننده.\nهم جدول حل کنید، هم حافظه خودتان را تقویت کنید، هم دوستانتان را به چالش بکشید.\nنگران تمام شدنِ جدول ها نباشید، مراحل زیاد بازی و جدول های استاندارد آن، به زودی تمام نخواهند شد.\n\nقابلیت های متفاوت این برنامه:\n- امکان اشتراک سوال با دوستان\n- صفحه کلید مخصوص جدول\n- چالش های هیجان انگیز و پر سکه\n-جایزه روزانه\n-قابلیت دعوت از دوستان به بازی\n- به رو...
1597,بازی فندق - بازی فکری کلمات,💡 با فندق به یک استاد واژگان تبدیل شوید! \nفقط با کشیدن دست بر روی حروف، کلمات بامعنی بسیاری بسازید و به یک استاد واژگان تبدیل شوید! \nبرای استاد شدن تلاش زیادی لازمه، پس وقتشه که رمز و رموز موجود در کلمات را بیابید و ذهن خود را چندین برابر ارتقا دهید.\n\n📜 ویژگی های فندق 📜 \n- ساده، آسان و گیم پلی اعتیاد آور! \n- بیش از 800 مرحله و هزاران کلمات مخفی منتظر شماست! \n- قابلیت بررسی مراحل قبل و کشف رمز و رموز آنها! \n- هر روز یک چالش جدید تجربه کنید! \n- از اعضای خانواده و دوستان خود برای کمک ب...
1596,آمیرزا,"« کسب عنوان برترین بازی سال 96 از نگاه مخاطبان» \n« بازی دیگری از سازندگان باقلوا»\n« پدیده سال 97 »\nعزیزان به دنیای سراسر لذت و سرگرمی "" آمیرزا "" خوش آمدید.\n\nآمیرزا بازی متفاوت ، سرگرم کننده و فوق العاده جذاب در سبک کلمات است که شما را به چالشی بزرگ و شیرین دعوت میکند. هدف بازی پیدا کردن کلمات مورد نظر از بین حروف بهم ریخته شده است در این بین شما باید هوش و ذکاوت خود را در بین بزرگان و میرزاهای عرصه کلمات محک بزنید و اعتبار خود را افزایش دهید ، از میرزا دلاک مشتمالیان گرفته تا میرزا عالم..."
1595,Филворды: поиск слов,"Игра Филворды - это несколько сотен уровней интеллектуального наслаждения. Вы ищете загаданные слова, а симпатичные монстры помогают вам. Сможете ли вы дойти до конца игры, заполнив все квадраты словами, а сосуды наполнить разноцветной жидкостью? \n\nЦелая куча уровней, спрятанных в бутылках!\n\nБолее 1000 уровней подготовлены специально для вас. В начале разгадывание головоломки может показаться несложным занятием. Но, по мере того, как количество букв увеличивается, угадывать слова станови..."
1594,"Учимся Читать, Спасаем Зверей. Учим буквы, азбука.","""Учимся читать, Спасаем зверей"" это развивающая игра - азбука абвгдейка и дошкольное обучение чтению по слогам и по буквам на русском и украинском языках. Подготовка к школе для детей 3 - 4 лет и старше. Учим буквы, слоги и учимся читать в процессе увлекательной обучающей игры для детей - дошкольников от 3 до 7 лет.\n\nЗлая Снежная Королева заморозила волшебные буквы алфавита и животные попали в ледяной плен. И только ваш умный малыш может их спасти, для чего он должен выучить буквы алфавита..."
1593,Слово по подсказке Ассоциации - Логические игры,"Популярная игра в ассоциации поможет вам расшевелить мозги при решении головоломок! Игра на логику, вам нужно угадать слово по 5 ассоциациям, поочередно открывая их. Например: по ассоциациям ""Звезд

In [ ]:
apps.search("")

,name,description
1599,دردشاتي - تعارف شات و زواج,"تطبيق دردشاتي هو أضخم وأكبر تطبيق تعارف عربي في العالم العربي !\n\nدردشاتي الذي يشمل أكثر من مليون مستخدم يبحثون عن صداقة تعارف وزواج بالاضافة الى زواج مسيار، وهو تطبيق التعارف العربي الأكبر والاضخم بعدد مسجلين حقيقيين من انحاء العالم العربي .\n\nنقدم لكم تطبيق التعارف الاقوى والاشهر في الشرق الاوسط تطبيق ""دردشاتي""، والمعد بصورة خاصة للمستخدمين العرب، حيث يشمل التطبيق مئات الاف المشتركين ، والمئات الجدد يوميا من شتى انحاء العالم العربي .\n\nالتطبيق تم برمجته وتصميمه بأفضل النواحي التقنية حيث..."
1598,جدول شرح در متن : جدولستان,اگر علاقه زیادی به جدول دارید و دوست دارید در هر مکانی جدول حل کنید، به جدولستان خوش آمدید.\nسرزمینی جذاب که پر است از جدول های سرگرم کننده.\nهم جدول حل کنید، هم حافظه خودتان را تقویت کنید، هم دوستانتان را به چالش بکشید.\nنگران تمام شدنِ جدول ها نباشید، مراحل زیاد بازی و جدول های استاندارد آن، به زودی تمام نخواهند شد.\n\nقابلیت های متفاوت این برنامه:\n- امکان اشتراک سوال با دوستان\n- صفحه کلید مخصوص جدول\n- چالش های هیجان انگیز و پر سکه\n-جایزه روزانه\n-قابلیت دعوت از دوستان به بازی\n- به رو...
1597,بازی فندق - بازی فکری کلمات,💡 با فندق به یک استاد واژگان تبدیل شوید! \nفقط با کشیدن دست بر روی حروف، کلمات بامعنی بسیاری بسازید و به یک استاد واژگان تبدیل شوید! \nبرای استاد شدن تلاش زیادی لازمه، پس وقتشه که رمز و رموز موجود در کلمات را بیابید و ذهن خود را چندین برابر ارتقا دهید.\n\n📜 ویژگی های فندق 📜 \n- ساده، آسان و گیم پلی اعتیاد آور! \n- بیش از 800 مرحله و هزاران کلمات مخفی منتظر شماست! \n- قابلیت بررسی مراحل قبل و کشف رمز و رموز آنها! \n- هر روز یک چالش جدید تجربه کنید! \n- از اعضای خانواده و دوستان خود برای کمک ب...
1596,آمیرزا,"« کسب عنوان برترین بازی سال 96 از نگاه مخاطبان» \n« بازی دیگری از سازندگان باقلوا»\n« پدیده سال 97 »\nعزیزان به دنیای سراسر لذت و سرگرمی "" آمیرزا "" خوش آمدید.\n\nآمیرزا بازی متفاوت ، سرگرم کننده و فوق العاده جذاب در سبک کلمات است که شما را به چالشی بزرگ و شیرین دعوت میکند. هدف بازی پیدا کردن کلمات مورد نظر از بین حروف بهم ریخته شده است در این بین شما باید هوش و ذکاوت خود را در بین بزرگان و میرزاهای عرصه کلمات محک بزنید و اعتبار خود را افزایش دهید ، از میرزا دلاک مشتمالیان گرفته تا میرزا عالم..."
1595,Филворды: поиск слов,"Игра Филворды - это несколько сотен уровней интеллектуального наслаждения. Вы ищете загаданные слова, а симпатичные монстры помогают вам. Сможете ли вы дойти до конца игры, заполнив все квадраты словами, а сосуды наполнить разноцветной жидкостью? \n\nЦелая куча уровней, спрятанных в бутылках!\n\nБолее 1000 уровней подготовлены специально для вас. В начале разгадывание головоломки может показаться несложным занятием. Но, по мере того, как количество букв увеличивается, угадывать слова станови..."
1594,"Учимся Читать, Спасаем Зверей. Учим буквы, азбука.","""Учимся читать, Спасаем зверей"" это развивающая игра - азбука абвгдейка и дошкольное обучение чтению по слогам и по буквам на русском и украинском языках. Подготовка к школе для детей 3 - 4 лет и старше. Учим буквы, слоги и учимся читать в процессе увлекательной обучающей игры для детей - дошкольников от 3 до 7 лет.\n\nЗлая Снежная Королева заморозила волшебные буквы алфавита и животные попали в ледяной плен. И только ваш умный малыш может их спасти, для чего он должен выучить буквы алфавита..."
1593,Слово по подсказке Ассоциации - Логические игры,"Популярная игра в ассоциации поможет вам расшевелить мозги при решении головоломок! Игра на логику, вам нужно угадать слово по 5 ассоциациям, поочередно открывая их. Например: по ассоциациям ""Звезда"", ""Вакуум"", ""Вселенная"", ""Гагарин"" и ""Бесконечность"" становится понятно, что загадано слово ""Космос"". Открывай слова поочередно и как только догадаешься, что загадано, пробуй давать ответ. Чем меньше слов будет открыто, тем больше монет ты сможешь заработать. \n\nВам встретятся увлекательные и ин..."
1592,Сканворды на русском,"""Сканворды на русском""- интереснейшая словесная головоломка, которая не заставит Вас скучать. От создателей Быстрых кроссвордов. Еще больше пересечений слов. Еще проще, веселее и 

## Individual reflection

<div class="alert alert-info">
    <strong>After you have solved the lab,</strong> write a <em>brief</em> reflection (max. one A4 page) on the question(s) below.  Remember:
    <ul>
        <li>You are encouraged to discuss this part with your lab partner, but you should each write up your reflection <strong>individually</strong>.</li>
        <li><strong>Do not put your answers in the notebook</strong>; upload them in the separate submission opportunity for the reflections on Lisam.</li>
    </ul>
</div>

1. In Problem 1, which token had the highest tf–idf score, which the lowest?  Based on your knowledge of how tf–idf works, how would you explain this result?
2. Based on your observations in Problem 4, which preprocessing steps do you think are the most appropriate for this "search engine" example?  Why?

**Congratulations on finishing this lab! 👍**

<div class="alert alert-info">
    
➡️ Before you submit, **make sure the notebook can be run from start to finish** without errors.  For this, _restart the kernel_ and _run all cells_ from top to bottom. In Jupyter Notebook version 7 or higher, you can do this via "Run$\rightarrow$Restart Kernel and Run All Cells..." in the menu (or the "⏩" button in the toolbar).

</div>

In [47]:
token_tfidf = [(t, v) for t, v in zip(tokens, vec)]
token_tfidf.sort(key=lambda x: -x[1], reverse=True)

for t, v in token_tfidf:
    print(f"{t}: {v}")

00: 0.0
000: 0.0
0000: 0.0
007: 0.0
01: 0.0
02: 0.0
03: 0.0
05: 0.0
06: 0.0
0771: 0.0
08: 0.0
080: 0.0
09: 0.0
0b6up9: 0.0
0ghz: 0.0
0kr: 0.0
10: 0.0
100: 0.0
1000: 0.0
10000000: 0.0
1000den: 0.0
100125201589035673885: 0.0
100539321026946285107: 0.0
100544026660028: 0.0
100k: 0.0
100s: 0.0
100th: 0.0
101: 0.0
1010: 0.0
1024: 0.0
1028: 0.0
1028호: 0.0
105: 0.0
107456884368395361653: 0.0
108: 0.0
1080p: 0.0
109: 0.0
10cm: 0.0
10k: 0.0
10m: 0.0
10s: 0.0
10th: 0.0
10x10: 0.0
11: 0.0
110: 0.0
111: 0.0
112: 0.0
113: 0.0
113681410008759: 0.0
115: 0.0
115000952828: 0.0
115005748829: 0.0
115005748929: 0.0
115521255825330665764: 0.0
115863800042796476976: 0.0
1177: 0.0
117783509607924: 0.0
118: 0.0
119: 0.0
11v11: 0.0
12: 0.0
120: 0.0
1200: 0.0
123: 0.0
123kidsfun: 0.0
123kidsfunapps: 0.0
123s: 0.0
124: 0.0
126: 0.0
1261472480647068: 0.0
128: 0.0
12x12: 0.0
12x15: 0.0
13: 0.0
130: 0.0
1300: 0.0
131072: 0.0
1395916567305340: 0.0
13m: 0.0
13th: 0.0
14: 0.0
140: 0.0
1400: 0.0
140827: 0.0
14135178489

In [48]:
token_tfidf = [(t, v) for t, v in zip(tokens, vec)]
token_tfidf.sort(key=lambda x: x[1], reverse=True)

for t, v in token_tfidf:
    print(f"{t}: {v}")

pancakes: 0.6539332651185913
pile: 0.5304701435508047
let: 0.2615287714771797
see: 0.2557630827415271
many: 0.23491959669849022
how: 0.21153246225085887
up: 0.17216837691451817
can: 0.13047602895910532
you: 0.10276923239718011
00: 0.0
000: 0.0
0000: 0.0
007: 0.0
01: 0.0
02: 0.0
03: 0.0
05: 0.0
06: 0.0
0771: 0.0
08: 0.0
080: 0.0
09: 0.0
0b6up9: 0.0
0ghz: 0.0
0kr: 0.0
10: 0.0
100: 0.0
1000: 0.0
10000000: 0.0
1000den: 0.0
100125201589035673885: 0.0
100539321026946285107: 0.0
100544026660028: 0.0
100k: 0.0
100s: 0.0
100th: 0.0
101: 0.0
1010: 0.0
1024: 0.0
1028: 0.0
1028호: 0.0
105: 0.0
107456884368395361653: 0.0
108: 0.0
1080p: 0.0
109: 0.0
10cm: 0.0
10k: 0.0
10m: 0.0
10s: 0.0
10th: 0.0
10x10: 0.0
11: 0.0
110: 0.0
111: 0.0
112: 0.0
113: 0.0
113681410008759: 0.0
115: 0.0
115000952828: 0.0
115005748829: 0.0
115005748929: 0.0
115521255825330665764: 0.0
115863800042796476976: 0.0
1177: 0.0
117783509607924: 0.0
118: 0.0
119: 0.0
11v11: 0.0
12: 0.0
120: 0.0
1200: 0.0
123: 0.0
123kidsfun: 0.0
123k


1. All tokens not in the vector has a score of 0, pancakes had the highest score. A tf-idf score is high for a token if the token appears in the vector but the token is uncommon amongst all documents, this is why you and can get low scores while pancakes which is uncommon gets a higher score. 


2. Jibberish words dont seem to impact the search as apps.search("jumping, is fe8 robfsots, sdfhi, eaf8") and searching for "jumping" gave the same result. With lemmatization we performed worse than without it, we didnt even find shuriken jump. And then we found alot of unrelated searches. But with lemmatization we found nothing when we searched for jumped, so there are downsides to both.